<a href="https://colab.research.google.com/github/Sprg72/Data-Engineer/blob/main/notepad/Pyspark_lab3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# step1
!pip install findspark pyspark


In [ ]:
# step2
import findspark
findspark.init()

In [ ]:
#step3
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('myapp').getOrCreate()

In [ ]:
sc = spark.sparkContext
sc

<SparkContext master=local[*] appName=myapp>



```
# input file : emp1.txt
101, amar,90000,m,11
102, amala,20000,f,12
103, ankit,40000,m,13
104, ankita,60000,f,13
105, anusha,110000,f,12
106, anuz,20000,m,11
107, akash,100000,m,12
108, siva,20000,m,14
109, sivani,30000,f,15
110, mani,30000,m,12
111, manisha,300000,f,13
112, sivam,200000,m,12
113, varun,200000,m,13

```


```
row level transformations.
   ---> updating existed fields, or creating new fields.

1. Name ---> first letter uppercase, remaining into lowercase

def firstUpper(x):
  x = x.strip().lower()
  fc = x[0].upper
  rc = x[1:]
  return fc+rc

2. tax (new)   ---> 10% on salary    ---> tax = sal * 0.1
3. hra (new    ---> 20% on salary    ---> hra = sal * 0.2
4. netsal(new) ---> netsal = sal + hra - tax
5. grade (new) --->
      >= 100000            ---> A
      >= 75k and < 100000  ---> B
      >= 50k and < 75k     ---> C
      <50k                 ---> D


def toGrade(sal):
    grade = 'D'
    if sal >= 100000:
        grade = 'A'
    elif sal >= 75000:
        grade = 'B'
    elif sal >= 50000:
        grade = 'C'
    return grade


6. 6. gender (update) --->
    'm' -> 'Male'
    'f' -> 'Female'
    remaining -> 'Invalid'

def toGender(gend):
    gend = gend.lower()
    if gend == 'm':
        gender = 'Male'
    elif gend == 'f':
        gender = 'Female'
    else:
        gender = 'Invalid'
    return gender

7. dname (new) -->
    if dno == 11 --> 'Marketing'
    if dno == 12 --> 'Hr'
    if dno == 13 --> 'Finance'
    remaining --> 'Other'

def toDname(dno):
    dinfo = {11: 'Marketing', 12: 'Hr', 13: 'Finance'}
    dname = dinfo.get(dno, 'Other')
    return dname


# include all above transformations into one single function.

def transform(line):  #
    # id,name,sal,gend,dno
    w = line.strip().lower().split(',')
    id = w[0]
    name = firstUpper(w[1])
    sal = int(w[2])
    tax = sal * 0.1
    hra = sal * 0.2
    net = sal + hra - tax
    grade = toGrade(sal)
    gender = toGender(w[-2])
    dno = int(w[-1])
    dname = toDname(dno)
    res = [id, name, sal, tax, hra, net, grade, gender, dno, dname]
    res = [str(v) for v in res]
    resline = ','.join(res)
    return resline


    
```




In [ ]:
def firstUpper(x):
  x = x.strip().lower()
  fc = x[0].upper()
  rc = x[1:]
  return fc+rc

In [ ]:
def toGrade(sal):
    grade = 'D'
    if sal >= 100000:
        grade = 'A'
    elif sal >= 75000:
        grade = 'B'
    elif sal >= 50000:
        grade = 'C'
    return grade

In [ ]:
def toGender(gend):
    gend = gend.lower()
    if gend == 'm':
        gender = 'Male'
    elif gend == 'f':
        gender = 'Female'
    else:
        gender = 'Invalid'
    return gender

In [ ]:
def toDname(dno):
    dinfo = {11: 'Marketing', 12: 'Hr', 13: 'Finance'}
    dname = dinfo.get(dno, 'Other')
    return dname

In [ ]:
def transform(line):  #
    # id,name,sal,gend,dno
    w = line.strip().lower().split(',')
    id = w[0]
    name = firstUpper(w[1])
    sal = int(w[2])
    tax = sal * 0.1
    hra = sal * 0.2
    net = sal + hra - tax
    grade = toGrade(sal)
    gender = toGender(w[-2])
    dno = int(w[-1])
    dname = toDname(dno)
    res = [id, name, sal, tax, hra, net, grade, gender, dno, dname]
    res = [str(v) for v in res]
    resline = ','.join(res)
    return resline


In [ ]:
transform('101, Siva, 90000,m, 13')

'101,Siva,90000,9000.0,18000.0,99000.0,B,Male,13,Finance'

In [ ]:
emp = sc.textFile('/content/emp1.txt')
emp.collect()


['101, amar,90000,m,11',
 '102, amala,20000,f,12',
 '103, ankit,40000,m,13',
 '104, ankita,60000,f,13',
 '105, anusha,110000,f,12',
 '106, anuz,20000,m,11',
 '107, akash,100000,m,12',
 '108, siva,20000,m,14',
 '109, sivani,30000,f,15',
 '110, mani,30000,m,12',
 '111, manisha,300000,f,13',
 '112, sivam,200000,m,12',
 '113, varun,200000,m,13']

In [ ]:
# emp_enriched= emp.map(lambda x : transform(x))
emp_enriched= emp.map(transform)
emp_enriched.collect()

['101,Amar,90000,9000.0,18000.0,99000.0,B,Male,11,Marketing',
 '102,Amala,20000,2000.0,4000.0,22000.0,D,Female,12,Hr',
 '103,Ankit,40000,4000.0,8000.0,44000.0,D,Male,13,Finance',
 '104,Ankita,60000,6000.0,12000.0,66000.0,C,Female,13,Finance',
 '105,Anusha,110000,11000.0,22000.0,121000.0,A,Female,12,Hr',
 '106,Anuz,20000,2000.0,4000.0,22000.0,D,Male,11,Marketing',
 '107,Akash,100000,10000.0,20000.0,110000.0,A,Male,12,Hr',
 '108,Siva,20000,2000.0,4000.0,22000.0,D,Male,14,Other',
 '109,Sivani,30000,3000.0,6000.0,33000.0,D,Female,15,Other',
 '110,Mani,30000,3000.0,6000.0,33000.0,D,Male,12,Hr',
 '111,Manisha,300000,30000.0,60000.0,330000.0,A,Female,13,Finance',
 '112,Sivam,200000,20000.0,40000.0,220000.0,A,Male,12,Hr',
 '113,Varun,200000,20000.0,40000.0,220000.0,A,Male,13,Finance']

In [ ]:
emp_enriched.getNumPartitions()

2

In [ ]:
emp_enriched.coalesce(1).saveAsTextFile('/content/emp_transform')

Py4JJavaError: An error occurred while calling o113.saveAsTextFile.
: org.apache.hadoop.mapred.FileAlreadyExistsException: Output directory file:/content/emp_transform already exists
	at org.apache.hadoop.mapred.FileOutputFormat.checkOutputSpecs(FileOutputFormat.java:131)
	at org.apache.spark.internal.io.HadoopMapRedWriteConfigUtil.assertConf(SparkHadoopWriter.scala:303)
	at org.apache.spark.internal.io.SparkHadoopWriter$.write(SparkHadoopWriter.scala:73)
	at org.apache.spark.rdd.PairRDDFunctions.$anonfun$saveAsHadoopDataset$1(PairRDDFunctions.scala:1094)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.scala:18)
	at org.apache.spark.rdd.RDDOperationScope$.withScope(RDDOperationScope.scala:151)
	at org.apache.spark.rdd.RDDOperationScope$.withScope(RDDOperationScope.scala:112)
	at org.apache.spark.rdd.RDD.withScope(RDD.scala:417)
	at org.apache.spark.rdd.PairRDDFunctions.saveAsHadoopDataset(PairRDDFunctions.scala:1092)
	at org.apache.spark.rdd.PairRDDFunctions.$anonfun$